In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import json
import shutil
import zipfile
import warnings

import ezc3d
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from scipy.signal import butter, sosfiltfilt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'obp_utils.py').is_file()), Path.cwd())
RAW = ROOT / 'data' / 'raw' / 'pitching'
OUT = ROOT / 'results' / 'head_trunk_release_consistency'
FIG = ROOT / 'figures' / 'head_trunk_release_consistency'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)

AUTO_DOWNLOAD = True
PRIMARY_MIN_PITCHES = 4
SENSITIVITY_MIN_PITCHES = (3, 5)
PRIMARY_HEAD_CUTOFF_HZ = 12
FILTER_SENSITIVITY_HZ = (8, 16)
PRE_RELEASE_MS = 100
MAX_MARKER_GAP_FRAMES = 3
MAX_HEAD_PAIR_DISTANCE_CHANGE_M = .035
BOOTSTRAPS = 1500
CV_SEEDS = (20261017, 20261018)
FOLDS = 5
INNER_FOLDS = 4
ALPHAS = (.1, 1., 10., 100., 1000.)
SEED = 20261017
SELECTED_SESSION = None

RELEASE_BASE = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/'
CSV_BASE = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_pitching/data/'
ARCHIVE_HASHES = {
    'pitching_c3d.zip': '2f5277dc63c7f7535fa4b15310d6f38b3f3b761e572896eec529854c8c920d29',
    'pitching_landmarks.zip': 'efcc078d5d8e77ed1fbf0b9f6a78f750c57a624165f71f30e34142b74db7ec22',
    'pitching_joint_angles.zip': 'b4c4854ffcc52a0f757e0b3a3c86acdc4a2fc2e50516a76e62aabdb916afed46',
}
HEAD_MARKERS = ('LFHD', 'RFHD', 'LBHD', 'RBHD')
PELVIS_MARKERS = ('LASI', 'RASI', 'LPSI', 'RPSI')
TORSO_COLUMNS = ('torso_angle_x', 'torso_angle_y', 'torso_angle_z')
HAND_COLUMNS = ('hand_jc_x', 'hand_jc_y', 'hand_jc_z')
TIME_GRID_MS = np.linspace(-PRE_RELEASE_MS, 0, 51)

print('Study 15 | Head & Trunk Stability and Release Consistency')
print('Primary outcome: within-session 2D (lateral/vertical) hand-joint-center release dispersion, cm.')
print('This is a release-position proxy, not a measured baseball release point or direct measure of command.')
print('Official OBP Dataset v1. Large C3D and processed archives download only if absent.')

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def download_file(url, destination):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'{destination} is missing. Download the official Dataset v1 files or enable AUTO_DOWNLOAD.')
    tmp = destination.with_name(destination.name + '.part')
    print(f'Downloading {destination.name} ...', flush=True)
    try:
        urlretrieve(url, tmp)
        tmp.replace(destination)
    finally:
        if tmp.exists():
            tmp.unlink()


def checked_archive(name):
    path = RAW / name
    if not path.exists():
        download_file(RELEASE_BASE + name, path)
    if file_sha256(path) != ARCHIVE_HASHES[name]:
        raise ValueError(f'{name}: SHA-256 does not match the official Dataset v1 release. Remove the corrupt archive.')
    return path


def ensure_processed_csv(name):
    target = RAW / name
    if target.exists():
        return target
    archive = checked_archive('pitching_' + name.replace('.csv', '.zip'))
    with zipfile.ZipFile(archive) as z:
        matches = [x for x in z.namelist() if Path(x).name == name and not x.endswith('/')]
        if len(matches) != 1:
            raise ValueError(f'Archive must contain exactly one {name}')
        tmp = target.with_suffix('.csv.part')
        with z.open(matches[0]) as f, tmp.open('wb') as w:
            shutil.copyfileobj(f, w)
        tmp.replace(target)
    return target


def ensure_c3d_files(filenames):
    root = RAW / 'c3d'
    root.mkdir(parents=True, exist_ok=True)
    existing = {p.name: p for p in root.rglob('*.c3d') if p.name in filenames}
    missing = set(filenames) - set(existing)
    if missing:
        archive = checked_archive('pitching_c3d.zip')
        with zipfile.ZipFile(archive) as z:
            for info in z.infolist():
                if info.is_dir() or Path(info.filename).name not in missing:
                    continue
                parts = Path(info.filename).parts
                if '..' in parts or Path(info.filename).is_absolute():
                    raise ValueError(f'Unsafe path in archive: {info.filename}')
                after = parts[parts.index('c3d') + 1:] if 'c3d' in parts else parts[-2:]
                destination = root.joinpath(*after)
                if not after or not str(destination.resolve()).startswith(str(root.resolve()) + str(Path('/'))):
                    raise ValueError(f'Unsafe C3D archive path: {info.filename}')
                if destination.name in existing:
                    raise ValueError(f'Duplicate C3D filename in archive: {destination.name}')
                destination.parent.mkdir(parents=True, exist_ok=True)
                with z.open(info) as f, destination.open('wb') as w:
                    shutil.copyfileobj(f, w)
                existing[destination.name] = destination
        missing = set(filenames) - set(existing)
    if missing:
        raise FileNotFoundError(f'Missing {len(missing)} C3D trials. Example: {sorted(missing)[:5]}')
    return {name: existing[name] for name in filenames}


for name, remote in [('metadata.csv', 'metadata.csv'), ('poi_metrics.csv', 'poi/poi_metrics.csv')]:
    if not (RAW / name).exists():
        download_file(CSV_BASE + remote, RAW / name)
ensure_processed_csv('joint_angles.csv')
ensure_processed_csv('landmarks.csv')

meta = pd.read_csv(RAW / 'metadata.csv', dtype={'user': str, 'session': str, 'session_pitch': str})
poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_pitch': str})
for label, data in [('metadata', meta), ('POI', poi)]:
    if data.session_pitch.duplicated().any():
        raise ValueError(f'Duplicate session_pitch in {label}')

meta['user'] = meta.user.map(lambda v: str(int(v)))
meta['session'] = meta.session.map(lambda v: str(int(v)))
meta['session_key'] = meta.user + '_' + meta.session
pitches = meta.merge(poi[['session_pitch', 'pitch_type', 'p_throws', 'pitch_speed_mph']],
                     on='session_pitch', how='inner', suffixes=('_metadata', ''), validate='one_to_one')
pitches = pitches.loc[pitches.pitch_type.eq('FF')].reset_index(drop=True)
if pitches.empty:
    raise ValueError('No FF pitches found')
if pitches.filename_new.duplicated().any():
    raise ValueError('Metadata contains duplicate C3D filenames')

angle_cols = ['session_pitch', 'time', 'BR_time', *TORSO_COLUMNS]
landmark_cols = ['session_pitch', 'time', 'BR_time', *HAND_COLUMNS]
angles = pd.read_csv(RAW / 'joint_angles.csv', usecols=angle_cols, dtype={'session_pitch': str})
landmarks = pd.read_csv(RAW / 'landmarks.csv', usecols=landmark_cols, dtype={'session_pitch': str})
for name, data in [('joint angles', angles), ('landmarks', landmarks)]:
    if data.duplicated(['session_pitch', 'time']).any():
        raise ValueError(f'Duplicate (session_pitch,time) in {name}')
angles_by_pitch = {k: g.sort_values('time') for k, g in angles.groupby('session_pitch', sort=False)}
landmarks_by_pitch = {k: g.sort_values('time') for k, g in landmarks.groupby('session_pitch', sort=False)}
files_by_pitch = ensure_c3d_files(set(pitches.filename_new))
print(f'{len(pitches)} fastballs, {pitches.user.nunique()} pitchers, {pitches.session_key.nunique()} sessions')
print('Available session pitch counts:')
display(pitches.groupby('session_key').size().value_counts().sort_index().rename('sessions').to_frame().T)

In [ ]:
def valid_time_grid(g, trial_count=None, sampling_hz=None):
    if g is None or len(g) < 25:
        raise ValueError('missing or short processed signal')
    t = g.time.to_numpy(float)
    if not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid processed time grid')
    if trial_count is not None:
        if len(t) != trial_count:
            raise ValueError(f'C3D/processed frame count mismatch: {trial_count} vs {len(t)}')
        if not np.allclose(t, np.arange(len(t)) / sampling_hz, atol=.00008, rtol=0):
            raise ValueError('C3D/processed time grids do not match')
    return t


def event_time(g):
    x = g.BR_time.to_numpy(float)
    if not np.isfinite(x).all() or np.max(x) - np.min(x) > .00005:
        raise ValueError('invalid or inconsistent ball release annotations')
    return float(x[0])


def interpolate_positions(times, signal, wanted):
    signal = np.asarray(signal, float)
    wanted = np.asarray(wanted, float)
    if (len(times) != len(signal) or len(times) < 2 or
            wanted.min() < times[0] - 1e-7 or wanted.max() > times[-1] + 1e-7):
        raise ValueError('requested timestamps are outside valid signal')
    if not np.isfinite(signal).all():
        raise ValueError('nonfinite processed kinematics in full trial')
    return np.column_stack([np.interp(wanted, times, signal[:, j]) for j in range(signal.shape[1])])


def fill_short_gaps(coords, max_gap=MAX_MARKER_GAP_FRAMES):
    x = np.array(coords, dtype=float, copy=True)
    bad = ~np.isfinite(x).all(axis=1)
    if bad.any():
        transitions = np.diff(np.r_[False, bad, False].astype(int))
        beginnings = np.flatnonzero(transitions == 1)
        endings = np.flatnonzero(transitions == -1)
        if any(a == 0 or b == len(x) or b - a > max_gap for a, b in zip(beginnings, endings)):
            raise ValueError('missing marker frames touch edge or exceed allowable gap')
        ix = np.flatnonzero(~bad)
        for axis in range(3):
            x[:, axis] = np.interp(np.arange(len(x)), ix, x[ix, axis])
    return x, int(bad.sum())


def c3d_markers(path):
    c = ezc3d.c3d(str(path))
    labels = [str(v).strip().upper() for v in c['parameters']['POINT']['LABELS']['value']]
    if any(labels.count(v) != 1 for v in HEAD_MARKERS):
        raise ValueError('C3D missing or duplicates a required head marker')
    unit = str(c['parameters']['POINT']['UNITS']['value'][0]).strip().lower()
    if unit not in ('m', 'cm', 'mm'):
        raise ValueError(f'Unsupported C3D units: {unit}')
    scale = {'m': 1., 'cm': .01, 'mm': .001}[unit]
    points = c['data']['points'][:3].transpose(2, 1, 0) * scale
    residuals = c['data']['meta_points']['residuals'][0].T
    invalid = ((residuals < 0) | ~np.isfinite(points).all(axis=2) |
               (np.linalg.norm(points, axis=2) == 0))
    points[invalid] = np.nan
    fs = float(c['header']['points']['frame_rate'])
    if fs <= 0 or not np.isfinite(fs):
        raise ValueError('invalid C3D frame rate')
    head = points[:, [labels.index(s) for s in HEAD_MARKERS], :]
    pelvis = (points[:, [labels.index(s) for s in PELVIS_MARKERS], :]
              if all(labels.count(s) == 1 for s in PELVIS_MARKERS) else None)
    return dict(head=head, pelvis=pelvis, fs=fs, frames=len(points))


def extract_head(trial, br, cutoff=PRIMARY_HEAD_CUTOFF_HZ):
    t = np.arange(trial['frames']) / trial['fs']
    start = br - .17
    finish = br + .07
    if start < t[0] or finish > t[-1]:
        raise ValueError('head filtering margin outside C3D capture')
    select = (t >= start) & (t <= finish)
    local_t = t[select]
    if len(local_t) < 40:
        raise ValueError('too few frames around release')
    repaired, counts = [], 0
    for i in range(4):
        x, filled = fill_short_gaps(trial['head'][select, i, :])
        repaired.append(x)
        counts += filled
    head = np.stack(repaired, axis=1)
    greatest_pair_change = max(float(np.ptp(np.linalg.norm(head[:, a] - head[:, b], axis=1)))
                               for a in range(4) for b in range(a + 1, 4))
    if greatest_pair_change > MAX_HEAD_PAIR_DISTANCE_CHANGE_M:
        raise ValueError('head marker geometry changes excessively around release')
    center = head.mean(axis=1)
    sos = butter(4, cutoff, fs=trial['fs'], output='sos')
    center_smooth = sosfiltfilt(sos, center, axis=0)
    wanted = br + TIME_GRID_MS / 1000
    head_trace = interpolate_positions(local_t, center_smooth, wanted)

    relative_trace = np.full_like(head_trace, np.nan)
    if trial['pelvis'] is not None:
        try:
            pel = np.stack([fill_short_gaps(trial['pelvis'][select, i, :])[0]
                            for i in range(4)], axis=1).mean(axis=1)
            pelvis_smooth = sosfiltfilt(sos, pel, axis=0)
            relative_trace = head_trace - interpolate_positions(local_t, pelvis_smooth, wanted)
        except (ValueError, IndexError):
            pass
    return head_trace, relative_trace, counts, greatest_pair_change


def processed_release_pitch(angle_g, landmark_g, trial):
    at = valid_time_grid(angle_g)
    lt = valid_time_grid(landmark_g, trial['frames'], trial['fs'])
    valid_time_grid(angle_g, trial['frames'], trial['fs'])
    br_a, br_l = event_time(angle_g), event_time(landmark_g)
    if abs(br_a - br_l) > .00005:
        raise ValueError('release annotations disagree between processed files')
    if br_a - PRE_RELEASE_MS / 1000 < max(at[0], lt[0]) or br_a > min(at[-1], lt[-1]):
        raise ValueError('missing complete pre-release kinematic window')
    wanted = br_a + TIME_GRID_MS / 1000
    angle_values = angle_g[list(TORSO_COLUMNS)].to_numpy(float)
    if not np.isfinite(angle_values).all():
        raise ValueError('nonfinite torso data')
    angle_values = np.rad2deg(np.unwrap(np.deg2rad(angle_values), axis=0))
    torso_trace = interpolate_positions(at, angle_values, wanted)
    hand_release = interpolate_positions(lt, landmark_g[list(HAND_COLUMNS)].to_numpy(float), [br_a])[0]
    return br_a, torso_trace, hand_release


pitch_records, pitch_audit, traces = [], [], {}
for i, row in enumerate(pitches.itertuples(index=False), 1):
    pitch_id = str(row.session_pitch)
    try:
        ag = angles_by_pitch.get(pitch_id)
        lg = landmarks_by_pitch.get(pitch_id)
        if ag is None or lg is None:
            raise ValueError('missing processed angle or landmark table')
        path = files_by_pitch[row.filename_new]
        trial = c3d_markers(path)
        br, torso, hand = processed_release_pitch(ag, lg, trial)
        head, head_rel, filled, pair_change = extract_head(trial, br)
        pitch_records.append(dict(session_pitch=pitch_id, user=row.user, session=row.session,
                                  session_key=row.session_key, playing_level=row.playing_level,
                                  height_m=float(row.session_height_m), speed_mph=float(row.pitch_speed_mph),
                                  p_throws=row.p_throws, br_time=br, frame_rate_hz=trial['fs'],
                                  marker_frames_filled=filled, head_pair_change_mm=1000 * pair_change,
                                  hand_x_m=hand[0], hand_y_m=hand[1], hand_z_m=hand[2],
                                  head_x_m=head[-1, 0], head_y_m=head[-1, 1], head_z_m=head[-1, 2],
                                  torso_x_deg=torso[-1, 0], torso_y_deg=torso[-1, 1], torso_z_deg=torso[-1, 2],
                                  head_rel_x_m=head_rel[-1, 0], head_rel_y_m=head_rel[-1, 1],
                                  head_rel_z_m=head_rel[-1, 2]))
        traces[pitch_id] = dict(head=head, head_rel=head_rel, torso=torso, raw_c3d=trial, br=br)
        status = 'retained'
    except (KeyError, ValueError, RuntimeError, OSError, IndexError) as exc:
        status = str(exc)
    pitch_audit.append(dict(session_pitch=pitch_id, filename=row.filename_new, status=status))
    if i % 100 == 0:
        print(f'Processed {i}/{len(pitches)} C3Ds')

pitch_audit = pd.DataFrame(pitch_audit)
pitch_audit.to_csv(OUT / 'pitch_eligibility_audit.csv', index=False)
display(pitch_audit.status.value_counts().rename('pitches').to_frame())
if not pitch_records:
    raise ValueError('No pitches passed extraction. Inspect pitch_eligibility_audit.csv before analysis.')
pitch_metrics = pd.DataFrame(pitch_records)
pitch_metrics.to_csv(OUT / 'pitch_release_features.csv', index=False)
print(f'Complete processed-trunk / C3D-head / hand-release matches: {len(pitch_metrics)} pitches')

In [ ]:
def euclidean_rms_sample(point_array, columns):
    values = np.asarray(point_array[:, columns], dtype=float)
    if len(values) < 2 or not np.isfinite(values).all():
        return np.nan
    return float(np.sqrt(np.var(values, axis=0, ddof=1).sum()))


def trajectory_repeatability(cube, start_relative=False):
    x = np.asarray(cube, float)
    if x.ndim != 3 or x.shape[0] < 2 or not np.isfinite(x).all():
        return np.nan
    if start_relative:
        x = x - x[:, :1, :]
    return float(np.sqrt(np.mean(np.sum(np.var(x, axis=0, ddof=1), axis=1))))


def summarize_session(g, cutoff=PRIMARY_HEAD_CUTOFF_HZ):
    n = len(g)
    if n < 2:
        return None
    ids = g.session_pitch.to_list()
    stacked_head = []
    stacked_relative = []
    stacked_torso = []
    for pid in ids:
        t = traces[pid]
        if cutoff == PRIMARY_HEAD_CUTOFF_HZ:
            head = t['head']
            relative = t['head_rel']
        else:
            head, relative, _, _ = extract_head(t['raw_c3d'], t['br'], cutoff)
        stacked_head.append(head)
        stacked_relative.append(relative)
        stacked_torso.append(t['torso'])
    head_cube = np.stack(stacked_head)
    relative_cube = np.stack(stacked_relative)
    torso_cube = np.stack(stacked_torso)

    hand_release = g[['hand_x_m', 'hand_y_m', 'hand_z_m']].to_numpy(float)
    head_release = head_cube[:, -1, :]
    torso_release = torso_cube[:, -1, :]
    base = dict(user=g.user.iloc[0], session=g.session.iloc[0], session_key=g.session_key.iloc[0],
                n_pitches=n, p_throws=g.p_throws.iloc[0], playing_level=g.playing_level.iloc[0],
                height_m=float(g.height_m.mean()), mean_speed_mph=float(g.speed_mph.mean()),
                hand_y_sd_cm=100 * np.std(hand_release[:, 1], ddof=1),
                hand_z_sd_cm=100 * np.std(hand_release[:, 2], ddof=1),
                hand_x_sd_cm=100 * np.std(hand_release[:, 0], ddof=1),
                hand_yz_rms_cm=100 * euclidean_rms_sample(hand_release, [1, 2]),
                hand_xyz_rms_cm=100 * euclidean_rms_sample(hand_release, [0, 1, 2]),
                head_release_yz_rms_cm=100 * euclidean_rms_sample(head_release, [1, 2]),
                head_release_xyz_rms_cm=100 * euclidean_rms_sample(head_release, [0, 1, 2]),
                head_pre100_abs_rms_cm=100 * trajectory_repeatability(head_cube),
                head_pre100_shape_rms_cm=100 * trajectory_repeatability(head_cube, start_relative=True),
                torso_release_sd_deg=euclidean_rms_sample(torso_release, [0, 1, 2]),
                torso_pre100_rms_deg=trajectory_repeatability(torso_cube),
                torso_pre100_shape_rms_deg=trajectory_repeatability(torso_cube, start_relative=True),
                torso_flexion_release_sd_deg=float(np.std(torso_release[:, 0], ddof=1)),
                torso_lateral_release_sd_deg=float(np.std(torso_release[:, 1], ddof=1)),
                torso_rotation_release_sd_deg=float(np.std(torso_release[:, 2], ddof=1)),
                head_pelvis_yz_rms_cm=100 * euclidean_rms_sample(relative_cube[:, -1, :], [1, 2]))
    return base


def session_table(pitch_table, minimum):
    values = []
    for _, g in pitch_table.groupby('session_key', sort=True):
        if len(g) >= minimum:
            value = summarize_session(g)
            if value is not None:
                values.append(value)
    return pd.DataFrame(values)


sessions_all = session_table(pitch_metrics, 2)
sessions = sessions_all.loc[sessions_all.n_pitches.ge(PRIMARY_MIN_PITCHES)].reset_index(drop=True)
if sessions.empty:
    raise ValueError(f'No sessions have at least {PRIMARY_MIN_PITCHES} valid pitches; see eligibility audit.')
for name, df in [('session_features_all_2plus.csv', sessions_all),
                 ('session_features_primary.csv', sessions)]:
    df.to_csv(OUT / name, index=False)
print(f'Primary sample: {len(sessions)} sessions, {sessions.user.nunique()} pitchers, '
      f'{sessions.n_pitches.sum()} matched pitches')
print('Sessions available for minimum counts:',
      {n: int(sessions_all.n_pitches.ge(n).sum()) for n in (3, 4, 5)})
display(sessions[['hand_yz_rms_cm', 'head_release_yz_rms_cm', 'torso_release_sd_deg',
                  'head_pre100_shape_rms_cm', 'torso_pre100_shape_rms_deg']].describe().round(3))

In [ ]:
def bootstrap_spearman(x, y, reps=BOOTSTRAPS, seed=SEED):
    d = pd.DataFrame({'x': x, 'y': y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) < 5 or d.x.nunique() < 2 or d.y.nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan, len(d)
    rho, pvalue = spearmanr(d.x, d.y)
    a = d.to_numpy(float)
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(reps):
        resampled = a[rng.integers(0, len(a), len(a))]
        if np.ptp(resampled[:, 0]) == 0 or np.ptp(resampled[:, 1]) == 0:
            continue
        draws.append(float(spearmanr(resampled[:, 0], resampled[:, 1]).statistic))
    low, high = np.quantile(draws, [.025, .975]) if draws else (np.nan, np.nan)
    return float(rho), float(pvalue), float(low), float(high), len(d)


comparisons = [
    ('Trunk release SD', 'torso_release_sd_deg'),
    ('Head release YZ RMS', 'head_release_yz_rms_cm'),
    ('Trunk pre-release trajectory RMS', 'torso_pre100_rms_deg'),
    ('Head pre-release shape RMS', 'head_pre100_shape_rms_cm'),
    ('Head-pelvis release YZ RMS', 'head_pelvis_yz_rms_cm'),
]
rows = []
for label, predictor in comparisons:
    rho, p, lo, hi, n = bootstrap_spearman(sessions[predictor], sessions.hand_yz_rms_cm)
    rows.append(dict(predictor=label, column=predictor, n_sessions=n,
                     spearman_rho=rho, p_value=p, bootstrap_low=lo, bootstrap_high=hi))
associations = pd.DataFrame(rows)
valid = associations.p_value.notna()
associations['fdr_q'] = np.nan
if valid.any():
    associations.loc[valid, 'fdr_q'] = multipletests(associations.loc[valid, 'p_value'], method='fdr_bh')[1]
associations.to_csv(OUT / 'primary_cohort_associations.csv', index=False)
display(associations[['predictor', 'n_sessions', 'spearman_rho', 'bootstrap_low',
                      'bootstrap_high', 'p_value', 'fdr_q']].round(3))

sensitivity = []
for minimum in (3, 4, 5):
    cohort = sessions_all.loc[sessions_all.n_pitches.ge(minimum)]
    for label, column in comparisons:
        rho, p, lo, hi, n = bootstrap_spearman(cohort[column], cohort.hand_yz_rms_cm,
                                              reps=600, seed=SEED + minimum)
        sensitivity.append(dict(min_pitches=minimum, metric=label, n_sessions=n,
                                rho=rho, ci_low=lo, ci_high=hi, p_value=p))
sensitivity_df = pd.DataFrame(sensitivity)
sensitivity_df.to_csv(OUT / 'session_size_sensitivity.csv', index=False)
display(sensitivity_df[['min_pitches', 'metric', 'n_sessions', 'rho', 'ci_low', 'ci_high']].round(3))

In [ ]:
MODEL_FEATURES = {
    'Controls': ('mean_speed_mph', 'height_m', 'n_pitches'),
    'Controls + trunk': ('mean_speed_mph', 'height_m', 'n_pitches', 'torso_release_sd_deg'),
    'Controls + head': ('mean_speed_mph', 'height_m', 'n_pitches', 'head_release_yz_rms_cm'),
    'Controls + trunk + head': ('mean_speed_mph', 'height_m', 'n_pitches',
                                'torso_release_sd_deg', 'head_release_yz_rms_cm'),
    'Controls + trajectories': ('mean_speed_mph', 'height_m', 'n_pitches',
                                'torso_release_sd_deg', 'head_release_yz_rms_cm',
                                'torso_pre100_shape_rms_deg', 'head_pre100_shape_rms_cm'),
}
TARGET = 'hand_yz_rms_cm'


def group_folds(groups, n_splits, seed):
    groups = np.asarray(groups)
    unique = np.unique(groups)
    if len(unique) < n_splits:
        raise ValueError('Not enough independent pitchers for cross-validation')
    shuffled = np.random.default_rng(seed).permutation(unique)
    for heldout in np.array_split(shuffled, n_splits):
        test = np.flatnonzero(np.isin(groups, heldout))
        train = np.flatnonzero(~np.isin(groups, heldout))
        yield train, test


def select_alpha(X, y, groups, seed):
    results = []
    for a in ALPHAS:
        errors = []
        for train, test in group_folds(groups, INNER_FOLDS, seed):
            pipeline = make_pipeline(StandardScaler(), Ridge(alpha=a))
            pipeline.fit(X[train], y[train])
            errors.extend(abs(y[test] - pipeline.predict(X[test])))
        results.append(float(np.mean(errors)))
    return ALPHAS[int(np.argmin(results))]


all_model_columns = sorted({v for cols in MODEL_FEATURES.values() for v in cols})
model_data = sessions[['session_key', 'user', TARGET] + all_model_columns].replace(
    [np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
if len(model_data) < 25 or model_data.user.nunique() < FOLDS:
    print('Insufficient complete sessions for stable model comparison; descriptive associations remain available.')
    predictions = pd.DataFrame()
    cv_summary = pd.DataFrame()
    model_contrasts = pd.DataFrame()
else:
    y = model_data[TARGET].to_numpy(float)
    ids = model_data.user.to_numpy()
    prediction_rows = []
    for seed in CV_SEEDS:
        for fold, (train, test) in enumerate(group_folds(ids, FOLDS, seed), 1):
            for name, cols in MODEL_FEATURES.items():
                X = model_data[list(cols)].to_numpy(float)
                alpha = select_alpha(X[train], y[train], ids[train], seed + fold * 101)
                model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
                model.fit(X[train], y[train])
                predicted = model.predict(X[test])
                for local_idx, estimate in zip(test, predicted):
                    prediction_rows.append(dict(seed=seed, fold=fold,
                                                session_key=model_data.session_key.iloc[local_idx],
                                                user=ids[local_idx], model=name,
                                                actual_cm=y[local_idx], predicted_cm=float(estimate),
                                                alpha=alpha))
    predictions = pd.DataFrame(prediction_rows)
    predictions['abs_error_cm'] = abs(predictions.actual_cm - predictions.predicted_cm)
    predictions.to_csv(OUT / 'nested_grouped_cv_predictions.csv', index=False)
    cv_summary = predictions.groupby('model').agg(n_predictions=('abs_error_cm', 'size'),
                  mae_cm=('abs_error_cm', 'mean'), chosen_alpha_median=('alpha', 'median')).reset_index()
    cv_summary = cv_summary.sort_values('mae_cm')
    cv_summary.to_csv(OUT / 'nested_grouped_cv_summary.csv', index=False)
    display(cv_summary.round(3))

    averaged = predictions.groupby(['user', 'session_key', 'model']).abs_error_cm.mean().unstack()
    rng = np.random.default_rng(SEED)
    model_contrasts = []
    for simpler, bigger in [('Controls', 'Controls + trunk'),
                            ('Controls', 'Controls + head'),
                            ('Controls + trunk', 'Controls + trunk + head'),
                            ('Controls + trunk + head', 'Controls + trajectories')]:
        differences = (averaged[simpler] - averaged[bigger]).groupby(level=0).mean().to_numpy(float)
        draws = rng.choice(differences, size=(BOOTSTRAPS, len(differences)), replace=True).mean(axis=1)
        low, high = np.quantile(draws, [.025, .975])
        model_contrasts.append(dict(baseline=simpler, augmented=bigger, n_pitchers=len(differences),
                                    mae_improvement_cm=np.mean(differences),
                                    ci_low_cm=low, ci_high_cm=high))
    model_contrasts = pd.DataFrame(model_contrasts)
    model_contrasts.to_csv(OUT / 'paired_cv_improvements.csv', index=False)
    print('Positive MAE improvement favors the augmented model; intervals resample pitchers.')
    display(model_contrasts.round(3))

    selected = ['torso_release_sd_deg', 'head_release_yz_rms_cm',
                'mean_speed_mph', 'height_m', 'n_pitches']
    centered = model_data[selected]
    normalized = (centered - centered.mean()) / centered.std(ddof=0).replace(0, np.nan)
    normalized = normalized.dropna(axis=1, how='all')
    fit = sm.OLS(model_data[TARGET], sm.add_constant(normalized, has_constant='add')).fit(cov_type='HC3')
    coefficients = pd.DataFrame({'coefficient_cm_per_1SD': fit.params,
                                 'ci_low': fit.conf_int()[0], 'ci_high': fit.conf_int()[1],
                                 'p_value_HC3': fit.pvalues})
    coefficients.index.name = 'predictor'
    coefficients.to_csv(OUT / 'adjusted_exploratory_ols_hc3.csv')
    print('Exploratory adjusted linear associations (HC3 uncertainty; NOT causal effects)')
    display(coefficients.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True)
for ax, xcol, label in zip(axes,
                           ('torso_release_sd_deg', 'head_release_yz_rms_cm'),
                           ('Trunk release variability (degrees)', 'Head release YZ RMS (cm)')):
    plot_data = sessions[[xcol, TARGET]].dropna()
    ax.scatter(plot_data[xcol], plot_data[TARGET], s=38, alpha=.7, edgecolor='none')
    ax.set_xlabel(label)
    ax.set_ylabel('Hand-release YZ RMS (cm)')
    ax.grid(alpha=.2)
    if len(plot_data) >= 5:
        r, p = spearmanr(plot_data[xcol], plot_data[TARGET])
        ax.set_title(f'Spearman rho={r:.2f} (n={len(plot_data)})')
fig.suptitle('Between-pitch stability vs. hand-position dispersion (session level)')
fig.tight_layout()
fig.savefig(FIG / '01_release_stability_associations.png', dpi=180)
plt.show()

if len(sessions) >= 2:
    low = sessions.sort_values(TARGET).iloc[0].session_key
    high = sessions.sort_values(TARGET).iloc[-1].session_key
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True)
    for ax, session_key, label in zip(axes, (low, high), ('Smallest observed spread', 'Largest observed spread')):
        g = pitch_metrics.loc[pitch_metrics.session_key.eq(session_key)]
        pts = g[['hand_y_m', 'hand_z_m']].to_numpy(float) * 100
        pts -= pts.mean(axis=0)
        ax.scatter(pts[:, 0], pts[:, 1], s=70)
        for j, (x, z) in enumerate(pts):
            ax.annotate(str(j + 1), (x, z), xytext=(4, 3), textcoords='offset points', fontsize=9)
        ax.axvline(0, color='gray', lw=1, ls=':')
        ax.axhline(0, color='gray', lw=1, ls=':')
        ax.set_aspect('equal', adjustable='box')
        rms = float(sessions.loc[sessions.session_key.eq(session_key), TARGET].iloc[0])
        ax.set_title(f'{label}\nSession {session_key}, RMS={rms:.1f} cm, n={len(g)}')
        ax.set_xlabel('Centered lateral Y (cm)')
        ax.set_ylabel('Centered vertical Z (cm)')
        ax.grid(alpha=.2)
    fig.suptitle('Illustrative session-level hand-joint-center release clusters')
    fig.tight_layout()
    fig.savefig(FIG / '02_hand_release_clusters.png', dpi=180)
    plt.show()

if not cv_summary.empty:
    comparison = cv_summary.sort_values('mae_cm')
    fig, ax = plt.subplots(figsize=(9.6, 4.5))
    ax.barh(comparison.model, comparison.mae_cm)
    ax.invert_yaxis()
    ax.set_xlabel('Nested pitcher-held-out MAE (cm), lower is better')
    ax.set_title('Predictive comparison: controls, trunk, head, and joint models')
    ax.grid(axis='x', alpha=.2)
    fig.tight_layout()
    fig.savefig(FIG / '03_nested_cv_model_comparison.png', dpi=180)
    plt.show()

In [ ]:
chosen = str(SELECTED_SESSION) if SELECTED_SESSION is not None else str(
    sessions.sort_values(['n_pitches', 'session_key'], ascending=[False, True]).session_key.iloc[0])
selected_pitches = pitch_metrics.loc[pitch_metrics.session_key.eq(chosen)]
if selected_pitches.empty or len(selected_pitches) < PRIMARY_MIN_PITCHES:
    raise ValueError(f'Selected session {chosen} does not have enough retained pitches')
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
for j, pitch in enumerate(selected_pitches.session_pitch):
    t = traces[pitch]
    head_cm = 100 * (t['head'][:, 2] - t['head'][0, 2])
    axes[0].plot(TIME_GRID_MS, head_cm, alpha=.75, label=str(pitch))
    axes[1].plot(TIME_GRID_MS, t['torso'][:, 1], alpha=.75, label=str(pitch))
axes[0].set_title('Head vertical displacement (from -100 ms)')
axes[1].set_title('Torso lateral tilt angle')
axes[0].set_ylabel('Vertical head movement (cm)')
axes[1].set_ylabel('Lateral tilt (degrees)')
for ax in axes:
    ax.set_xlabel('Time relative to ball release (ms)')
    ax.axvline(0, ls=':', lw=1, color='gray')
    ax.grid(alpha=.2)
axes[1].legend(title='Pitch ID', fontsize=8)
fig.suptitle(f'Release-aligned pre-release trajectories | session {chosen}')
fig.tight_layout()
fig.savefig(FIG / '04_release_aligned_trajectories.png', dpi=180)
plt.show()

filter_rows = []
for cutoff in (PRIMARY_HEAD_CUTOFF_HZ, *FILTER_SENSITIVITY_HZ):
    for session_key, g in pitch_metrics.groupby('session_key'):
        if len(g) < PRIMARY_MIN_PITCHES:
            continue
        try:
            stats = summarize_session(g, cutoff=cutoff)
            filter_rows.append(dict(cutoff_hz=cutoff, session_key=session_key,
                                    head_release_yz_rms_cm=stats['head_release_yz_rms_cm'],
                                    head_pre100_shape_rms_cm=stats['head_pre100_shape_rms_cm']))
        except (ValueError, KeyError, IndexError):
            pass
filter_summary = pd.DataFrame(filter_rows)
filter_summary.to_csv(OUT / 'head_filter_sensitivity.csv', index=False)
if not filter_summary.empty:
    for metric in ('head_release_yz_rms_cm', 'head_pre100_shape_rms_cm'):
        wide = filter_summary.pivot(index='session_key', columns='cutoff_hz', values=metric)
        if PRIMARY_HEAD_CUTOFF_HZ in wide:
            for alternate in FILTER_SENSITIVITY_HZ:
                if alternate in wide:
                    pair = wide[[PRIMARY_HEAD_CUTOFF_HZ, alternate]].dropna()
                    if len(pair) > 3:
                        print(f'{metric}: {PRIMARY_HEAD_CUTOFF_HZ} vs {alternate} Hz, '
                              f'{len(pair)} matched sessions, rank correlation = '
                              f'{spearmanr(pair.iloc[:, 0], pair.iloc[:, 1]).statistic:.3f}')

In [ ]:
DIAG = OUT / 'focused_pelvis_relative_diagnostic'
DIAG_FIG = FIG / 'focused_pelvis_relative_diagnostic'
for directory in (DIAG, DIAG_FIG):
    directory.mkdir(parents=True, exist_ok=True)

DIAG_BOOTSTRAPS = 800
DIAG_SEED = 20261019
DIAG_SENSITIVITY_MIN = (3, 4, 5)
DIAG_SHOW_N_OUTLIERS = 8

print('FOCUSED DIAGNOSTIC | Shared translation, pelvis-relative release, axis separation, influence')
print('Readout 1: absolute head vs hand dispersion, then head relative to pelvis vs absolute hand dispersion.')
print('Readout 2: both head and hand relative to the same pelvis reference (shared-anchor caution).')
print('Readout 3: lateral Y and vertical Z, held-out models and leave-one-pitcher-out influence.')
print('All results are observational and refer to the throwing-hand joint center, not pitch target accuracy.')

In [ ]:
def diag_sd_cm(a):
    values = np.asarray(a, float)
    if len(values) < 2 or not np.isfinite(values).all():
        return np.nan
    return float(100 * np.std(values, ddof=1))


def diag_rms_yz_cm(x):
    v = np.asarray(x, float)
    if v.ndim != 2 or v.shape[1] != 3 or len(v) < 2 or not np.isfinite(v).all():
        return np.nan
    return float(100 * np.sqrt(np.var(v[:, 1:3], axis=0, ddof=1).sum()))


def focused_pitch_table(original, stored_traces):
    records = []
    audit = []
    for row in original.itertuples(index=False):
        pid = str(row.session_pitch)
        try:
            trace = stored_traces[pid]
            head = np.asarray(trace['head'], float)[-1]
            head_relative = np.asarray(trace['head_rel'], float)[-1]
            if head.shape != (3,) or head_relative.shape != (3,):
                raise ValueError('missing or invalid C3D head/pelvis release coordinates')
            if not np.isfinite(np.r_[head, head_relative]).all():
                raise ValueError('C3D pelvis markers unavailable or did not pass quality controls')
            pelvis = head - head_relative
            hand = np.array([row.hand_x_m, row.hand_y_m, row.hand_z_m], dtype=float)
            if not np.isfinite(hand).all():
                raise ValueError('missing processed hand release position')
            head_pelvis_dist = float(np.linalg.norm(head_relative))
            hand_pelvis_dist = float(np.linalg.norm(hand - pelvis))
            if head_pelvis_dist > 2.0 or hand_pelvis_dist > 2.0:
                raise ValueError('implausible common-frame head/pelvis or hand/pelvis separation (>2 m)')
            result = dict(session_pitch=pid, user=str(row.user), session_key=str(row.session_key),
                          p_throws=row.p_throws, playing_level=row.playing_level,
                          speed_mph=float(row.speed_mph), height_m=float(row.height_m),
                          head_pelvis_separation_m=head_pelvis_dist,
                          hand_pelvis_separation_m=hand_pelvis_dist)
            for i, axis in enumerate('xyz'):
                result.update({'head_' + axis + '_m': head[i],
                               'pelvis_' + axis + '_m': pelvis[i],
                               'hand_' + axis + '_m': hand[i],
                               'head_rel_' + axis + '_m': head_relative[i],
                               'hand_rel_' + axis + '_m': hand[i] - pelvis[i]})
            records.append(result)
            audit.append(dict(session_pitch=pid, status='retained'))
        except (KeyError, ValueError, IndexError, TypeError) as exc:
            audit.append(dict(session_pitch=pid, status=str(exc)))
    return pd.DataFrame(records), pd.DataFrame(audit)


def focused_session_table(table, minimum=2):
    summary = []
    positions = ('head', 'hand', 'pelvis', 'head_rel', 'hand_rel')
    for session_key, grp in table.groupby('session_key', sort=True):
        if len(grp) < minimum:
            continue
        q = dict(session_key=session_key, user=str(grp.user.iloc[0]), n_pitches=len(grp),
                 height_m=float(grp.height_m.mean()), mean_speed_mph=float(grp.speed_mph.mean()))
        for prefix in positions:
            arr = grp[[f'{prefix}_{a}_m' for a in 'xyz']].to_numpy(float)
            q[prefix + '_yz_rms_cm'] = diag_rms_yz_cm(arr)
            for j, axis in enumerate('xyz'):
                q[f'{prefix}_{axis}_sd_cm'] = diag_sd_cm(arr[:, j])
        hand = grp[[f'hand_{a}_m' for a in 'xyz']].to_numpy(float)
        pelvis = grp[[f'pelvis_{a}_m' for a in 'xyz']].to_numpy(float)
        q['hand_pelvis_yz_cov_cm2'] = float(10000 * sum(
            np.cov(hand[:, j], pelvis[:, j], ddof=1)[0, 1] for j in (1, 2)))
        q['torso_release_sd_deg'] = float(euclidean_rms_sample(
            np.stack([stored_traces[p]['torso'][-1] for p in grp.session_pitch]), [0, 1, 2]))
        q['head_rel_pre100_shape_rms_cm'] = float(100 * trajectory_repeatability(
            np.stack([stored_traces[p]['head_rel'] for p in grp.session_pitch]), start_relative=True))
        summary.append(q)
    return pd.DataFrame(summary)

stored_traces = traces
focused_pitches, focused_pitch_audit = focused_pitch_table(pitch_metrics, stored_traces)
focused_pitch_audit.to_csv(DIAG / '01_pitch_audit.csv', index=False)
focused_pitches.to_csv(DIAG / '02_pelvis_relative_pitch_positions.csv', index=False)
if focused_pitches.empty:
    raise ValueError('No pitching C3Ds provided valid head AND pelvis markers; review the pitch audit.')
focused_all = focused_session_table(focused_pitches, minimum=2)
focused = focused_all.loc[focused_all.n_pitches.ge(PRIMARY_MIN_PITCHES)].reset_index(drop=True)
if len(focused) < 15:
    raise ValueError('Insufficient eligible primary sessions for the focused analysis.')
focused_all.to_csv(DIAG / '03_matched_session_metrics_2plus.csv', index=False)
focused.to_csv(DIAG / '04_matched_session_metrics_primary.csv', index=False)

matched = focused.merge(sessions[['session_key', 'hand_yz_rms_cm', 'head_release_yz_rms_cm']],
                        on='session_key', how='left', validate='one_to_one', suffixes=('', '_prior'))
if len(matched) == len(sessions) and matched[['hand_yz_rms_cm', 'head_release_yz_rms_cm']].notna().all().all():
    np.testing.assert_allclose(matched.hand_yz_rms_cm, matched.hand_yz_rms_cm_prior, atol=1e-8)
    np.testing.assert_allclose(matched.head_yz_rms_cm, matched.head_release_yz_rms_cm, atol=1e-8)

print(f'Pelvis-relative complete trials: {len(focused_pitches)}/{len(pitch_metrics)}')
print(f'Matched sessions at >= {PRIMARY_MIN_PITCHES} pitches: {len(focused)}; '
      f'original primary sessions: {len(sessions)}')
print('Sessions by count cutoff:', {n: int(focused_all.n_pitches.ge(n).sum()) for n in DIAG_SENSITIVITY_MIN})
display(focused[['head_yz_rms_cm', 'head_rel_yz_rms_cm', 'pelvis_yz_rms_cm',
                 'hand_yz_rms_cm', 'hand_rel_yz_rms_cm']].describe().round(3))

In [ ]:
DIAG_COMPARISONS = [
    ('Global head / global hand', 'head', 'hand'),
    ('Pelvis-relative head / global hand', 'head_rel', 'hand'),
    ('Global head / pelvis-relative hand', 'head', 'hand_rel'),
    ('Pelvis-relative head / pelvis-relative hand', 'head_rel', 'hand_rel'),
    ('Pelvis translation / global hand', 'pelvis', 'hand'),
    ('Pelvis translation / pelvis-relative hand', 'pelvis', 'hand_rel'),
]

def diag_correlations(table, minimum, bootstraps=DIAG_BOOTSTRAPS):
    cohort = table.loc[table.n_pitches.ge(minimum)]
    records = []
    for axis in ('yz_rms', 'y_sd', 'z_sd'):
        for j, (name, predictor, outcome) in enumerate(DIAG_COMPARISONS):
            x_col, y_col = predictor + '_' + axis + '_cm', outcome + '_' + axis + '_cm'
            rho, p, low, high, count = bootstrap_spearman(
                cohort[x_col], cohort[y_col], reps=bootstraps,
                seed=DIAG_SEED + j + 100 * ('yz_rms', 'y_sd', 'z_sd').index(axis) + minimum)
            records.append(dict(min_pitches=minimum, axis=axis, comparison=name,
                                predictor=x_col, outcome=y_col, n_sessions=count,
                                rho=rho, p_value=p, ci_low=low, ci_high=high))
    return pd.DataFrame(records)

primary_diag_correlations = diag_correlations(focused_all, PRIMARY_MIN_PITCHES)
primary_diag_correlations['exploratory_fdr_q'] = np.nan
valid = primary_diag_correlations.p_value.notna()
primary_diag_correlations.loc[valid, 'exploratory_fdr_q'] = multipletests(
    primary_diag_correlations.loc[valid, 'p_value'], method='fdr_bh')[1]
primary_diag_correlations.to_csv(DIAG / '05_matched_associations_by_axis.csv', index=False)
print('Global vs pelvis-relative associations | all on the EXACT same retained pitches and sessions')
display(primary_diag_correlations.query("axis == 'yz_rms'")[
    ['comparison', 'n_sessions', 'rho', 'ci_low', 'ci_high', 'p_value']].round(3))
print('Lateral and vertical outcomes are shown separately below; all p-values are exploratory.')
display(primary_diag_correlations.query("axis != 'yz_rms'")[
    ['axis', 'comparison', 'rho', 'ci_low', 'ci_high']].round(3))

sample_sensitivity = pd.concat([diag_correlations(focused_all, cutoff, bootstraps=300)
    for cutoff in DIAG_SENSITIVITY_MIN], ignore_index=True)
sample_sensitivity.to_csv(DIAG / '06_pitches_per_session_sensitivity.csv', index=False)
print('Sample-size sensitivity for the four primary coordinate specifications (YZ only):')
display(sample_sensitivity.query("axis == 'yz_rms' and comparison != 'Pelvis translation / global hand' "
                                  "and comparison != 'Pelvis translation / pelvis-relative hand'")[
    ['min_pitches', 'comparison', 'n_sessions', 'rho', 'ci_low', 'ci_high']].round(3))

variance_identity = (focused.hand_yz_rms_cm ** 2 + focused.pelvis_yz_rms_cm ** 2
                     - 2 * focused.hand_pelvis_yz_cov_cm2)
np.testing.assert_allclose(focused.hand_rel_yz_rms_cm ** 2,
                           variance_identity, rtol=1e-8, atol=1e-8)
print('Verified: Var(hand - pelvis) = Var(hand) + Var(pelvis) - 2 Cov(hand, pelvis) on Y and Z.')
print('Both pelvis-relative metrics share the same subtraction and are not independent measurements.')

In [ ]:
DIAG_FEATURES = {
    'Controls': ('mean_speed_mph', 'height_m', 'n_pitches'),
    '+ Trunk': ('mean_speed_mph', 'height_m', 'n_pitches', 'torso_release_sd_deg'),
    '+ Pelvis': ('mean_speed_mph', 'height_m', 'n_pitches', 'pelvis_yz_rms_cm'),
    '+ Pelvis + trunk': ('mean_speed_mph', 'height_m', 'n_pitches', 'pelvis_yz_rms_cm', 'torso_release_sd_deg'),
    '+ Pelvis + trunk + abs head': ('mean_speed_mph', 'height_m', 'n_pitches', 'pelvis_yz_rms_cm',
                                     'torso_release_sd_deg', 'head_yz_rms_cm'),
    '+ Pelvis + trunk + rel head': ('mean_speed_mph', 'height_m', 'n_pitches', 'pelvis_yz_rms_cm',
                                     'torso_release_sd_deg', 'head_rel_yz_rms_cm'),
    '+ Pelvis + trunk + rel head shape': ('mean_speed_mph', 'height_m', 'n_pitches', 'pelvis_yz_rms_cm',
                                     'torso_release_sd_deg', 'head_rel_pre100_shape_rms_cm'),
}
DIAG_TARGETS = {'global_hand': 'hand_yz_rms_cm', 'pelvis_relative_hand': 'hand_rel_yz_rms_cm'}

all_diag_features = sorted(set(c for columns in DIAG_FEATURES.values() for c in columns))
complete_diag = focused[['user', 'session_key', *DIAG_TARGETS.values(), *all_diag_features]].replace(
    [np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
if len(complete_diag) < 25 or complete_diag.user.nunique() < FOLDS:
    raise ValueError('Insufficient complete matched pitcher sessions for diagnostic nested CV.')
if len(complete_diag) != len(focused):
    print(f'CV complete cases: {len(complete_diag)} of {len(focused)} primary sessions')

rows = []
groups = complete_diag.user.to_numpy()
for target_label, target_col in DIAG_TARGETS.items():
    y = complete_diag[target_col].to_numpy(float)
    for seed in CV_SEEDS:
        for fold, (train, test) in enumerate(group_folds(groups, FOLDS, seed), 1):
            for model_label, columns in DIAG_FEATURES.items():
                X = complete_diag[list(columns)].to_numpy(float)
                chosen_alpha = select_alpha(X[train], y[train], groups[train], seed + fold * 101)
                reg = make_pipeline(StandardScaler(), Ridge(alpha=chosen_alpha))
                reg.fit(X[train], y[train])
                for index, prediction in zip(test, reg.predict(X[test])):
                    rows.append(dict(target=target_label, seed=seed, fold=fold,
                                     user=groups[index], session_key=complete_diag.session_key.iloc[index],
                                     model=model_label, observed_cm=y[index], predicted_cm=float(prediction),
                                     alpha=chosen_alpha))

focused_predictions = pd.DataFrame(rows)
focused_predictions['absolute_error_cm'] = (focused_predictions.observed_cm - focused_predictions.predicted_cm).abs()
focused_predictions.to_csv(DIAG / '07_nested_grouped_cv_predictions.csv', index=False)
focused_cv = focused_predictions.groupby(['target', 'model']).agg(
    n_predictions=('absolute_error_cm', 'size'), mae_cm=('absolute_error_cm', 'mean'),
    median_alpha=('alpha', 'median')).reset_index().sort_values(['target', 'mae_cm'])
focused_cv.to_csv(DIAG / '08_nested_grouped_cv_summary.csv', index=False)
print('Nested validation: same complete sessions, same folds, separate global vs pelvis-relative outcomes.')
print('MAEs are compared only WITHIN each outcome definition, not across the two outcomes.')
display(focused_cv.round(3))

DIAG_CONTRASTS = [
    ('Controls', '+ Trunk'), ('Controls', '+ Pelvis'),
    ('+ Pelvis + trunk', '+ Pelvis + trunk + abs head'),
    ('+ Pelvis + trunk', '+ Pelvis + trunk + rel head'),
    ('+ Pelvis + trunk', '+ Pelvis + trunk + rel head shape'),
]

def paired_model_comparisons(prediction_frame, n_boot=DIAG_BOOTSTRAPS):
    records = []
    per_pitcher = (prediction_frame.groupby(['target', 'user', 'model'])
                   .absolute_error_cm.mean().reset_index())
    rng = np.random.default_rng(DIAG_SEED)
    for target, subgroup in per_pitcher.groupby('target'):
        wide = subgroup.pivot(index='user', columns='model', values='absolute_error_cm')
        for baseline, augmented in DIAG_CONTRASTS:
            delta = (wide[baseline] - wide[augmented]).dropna().to_numpy(float)
            if len(delta) < 5:
                continue
            draws = rng.choice(delta, size=(n_boot, len(delta)), replace=True).mean(axis=1)
            low, high = np.quantile(draws, [.025, .975])
            records.append(dict(target=target, baseline=baseline, augmented=augmented,
                                n_pitchers=len(delta), improvement_cm=float(delta.mean()),
                                ci_low_cm=float(low), ci_high_cm=float(high)))
    return pd.DataFrame(records)

focused_contrasts = paired_model_comparisons(focused_predictions)
focused_contrasts.to_csv(DIAG / '09_matched_paired_cv_improvements.csv', index=False)
print('Positive paired improvement favors the augmented model. Each pitcher is weighted once.')
display(focused_contrasts.round(3))

In [ ]:
def adjusted_diagnostic(table, target, candidate):
    columns = ['mean_speed_mph', 'height_m', 'n_pitches',
               'torso_release_sd_deg', 'pelvis_yz_rms_cm', candidate]
    data = table[[target] + columns].replace([np.inf, -np.inf], np.nan).dropna()
    X = data[columns].copy()
    sigma = X.std(ddof=0)
    if (sigma <= 0).any() or len(data) <= len(columns) + 5:
        return None
    X = (X - X.mean()) / sigma
    model = sm.OLS(data[target], sm.add_constant(X, has_constant='add')).fit(cov_type='HC3')
    limits = model.conf_int()
    return dict(outcome=target, head_metric=candidate, n_sessions=len(data),
                standardized_head_coefficient_cm=float(model.params[candidate]),
                ci_low_cm=float(limits.loc[candidate, 0]),
                ci_high_cm=float(limits.loc[candidate, 1]),
                head_pvalue_HC3=float(model.pvalues[candidate]),
                model_r2=float(model.rsquared),
                condition_number=float(np.linalg.cond(sm.add_constant(X, has_constant='add').to_numpy(float))))

adjusted_rows = []
for target in DIAG_TARGETS.values():
    for head_metric in ('head_yz_rms_cm', 'head_rel_yz_rms_cm', 'head_rel_pre100_shape_rms_cm'):
        result = adjusted_diagnostic(complete_diag, target, head_metric)
        if result is not None:
            adjusted_rows.append(result)
focused_adjusted = pd.DataFrame(adjusted_rows)
focused_adjusted.to_csv(DIAG / '10_matched_adjusted_ols_hc3.csv', index=False)
print('Adjusted head associations: all models include speed, body height, n pitches, pelvis spread, trunk spread.')
print('Coefficients are cm change in the outcome per 1 SD of candidate predictor; HC3 intervals; not causal.')
display(focused_adjusted.round(3))

In [ ]:
LOPO_ASSOCIATIONS = [
    ('abs_to_global', 'head_yz_rms_cm', 'hand_yz_rms_cm'),
    ('rel_to_global', 'head_rel_yz_rms_cm', 'hand_yz_rms_cm'),
    ('rel_to_rel', 'head_rel_yz_rms_cm', 'hand_rel_yz_rms_cm'),
    ('pelvis_to_global', 'pelvis_yz_rms_cm', 'hand_yz_rms_cm'),
]

def simple_rho(frame, x, y):
    data = frame[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 5 or data[x].nunique() < 2 or data[y].nunique() < 2:
        return np.nan
    return float(spearmanr(data[x], data[y]).statistic)

lopo_records = []
for removed in complete_diag.user.unique():
    smaller = complete_diag.loc[complete_diag.user.ne(removed)].copy()
    row = dict(omitted_pitcher=str(removed), retained_pitchers=smaller.user.nunique())
    for name, x, y in LOPO_ASSOCIATIONS:
        row['rho_' + name] = simple_rho(smaller, x, y)
    adj = adjusted_diagnostic(smaller, 'hand_yz_rms_cm', 'head_rel_yz_rms_cm')
    row['adjusted_rel_head_coefficient_cm'] = (adj['standardized_head_coefficient_cm']
                                               if adj is not None else np.nan)
    lopo_records.append(row)
lopo_associations = pd.DataFrame(lopo_records)
lopo_associations.to_csv(DIAG / '11_leave_one_pitcher_out_associations.csv', index=False)

per_pitcher_mae = focused_predictions.groupby(['target', 'user', 'model']).absolute_error_cm.mean().reset_index()
lopo_cv_rows = []
for (target, user), _ in per_pitcher_mae.groupby(['target', 'user']):
    remaining = per_pitcher_mae.query('target == @target and user != @user')
    wide = remaining.pivot(index='user', columns='model', values='absolute_error_cm')
    for baseline, augmented in DIAG_CONTRASTS:
        improvement = (wide[baseline] - wide[augmented]).dropna().mean()
        lopo_cv_rows.append(dict(target=target, omitted_pitcher=str(user),
                                 baseline=baseline, augmented=augmented,
                                 improvement_cm_excluding_pitcher=float(improvement)))
lopo_cv = pd.DataFrame(lopo_cv_rows)
lopo_cv.to_csv(DIAG / '12_leave_one_pitcher_out_cv_evaluation_influence.csv', index=False)

main_labels = ('rho_abs_to_global', 'rho_rel_to_global', 'rho_rel_to_rel',
               'rho_pelvis_to_global', 'adjusted_rel_head_coefficient_cm')
print('Full-sample vs leave-one-pitcher-out ranges (all adjusted regressions RE-FITTED):')
full_adj = adjusted_diagnostic(complete_diag, 'hand_yz_rms_cm', 'head_rel_yz_rms_cm')
full = [simple_rho(complete_diag, x, y) for _, x, y in LOPO_ASSOCIATIONS]
full.append(full_adj['standardized_head_coefficient_cm'] if full_adj else np.nan)
range_summary = pd.DataFrame([dict(metric=label, full_sample=value,
                                   smallest_after_omission=lopo_associations[label].min(),
                                   largest_after_omission=lopo_associations[label].max())
                              for label, value in zip(main_labels, full)])
range_summary.to_csv(DIAG / '13_influence_range_summary.csv', index=False)
display(range_summary.round(3))

lopo_associations['abs_to_global_change'] = abs(lopo_associations.rho_abs_to_global - full[0])
lopo_associations['rel_to_global_change'] = abs(lopo_associations.rho_rel_to_global - full[1])
print('Largest leave-one-pitcher-out changes in relative-head / absolute-hand correlation:')
display(lopo_associations.sort_values('rel_to_global_change', ascending=False)[
    ['omitted_pitcher', 'rho_abs_to_global', 'rho_rel_to_global',
     'rho_rel_to_rel', 'adjusted_rel_head_coefficient_cm']].head(DIAG_SHOW_N_OUTLIERS).round(3))

print('Paired CV influence (post-hoc omission from EXISTING held-out predictions, not nested-model refitting):')
influence_summary = (lopo_cv.groupby(['target', 'baseline', 'augmented'])
                     .improvement_cm_excluding_pitcher.agg(['min', 'max']).reset_index())
influence_summary.to_csv(DIAG / '14_cv_evaluation_influence_ranges.csv', index=False)
display(influence_summary.round(3))
print('LOPO evaluation checks the effect of a pitcher on reported error; it does not re-train CV models.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 9.0))
visual_pairs = [
    ('head_yz_rms_cm', 'hand_yz_rms_cm', 'Global head vs global hand'),
    ('head_rel_yz_rms_cm', 'hand_yz_rms_cm', 'Head relative to pelvis vs global hand'),
    ('head_rel_yz_rms_cm', 'hand_rel_yz_rms_cm', 'Both relative to pelvis (shared reference)'),
    ('pelvis_yz_rms_cm', 'hand_yz_rms_cm', 'Pelvis spread vs global hand'),
]
for ax, (xcol, ycol, label) in zip(axes.ravel(), visual_pairs):
    data = focused[[xcol, ycol]].dropna()
    ax.scatter(data[xcol], data[ycol], s=35, alpha=.75)
    rho = simple_rho(data, xcol, ycol)
    ax.set_title(f'{label}\nSpearman rho={rho:.3f}, n={len(data)}', fontsize=10)
    ax.set_xlabel(xcol.replace('_', ' '))
    ax.set_ylabel(ycol.replace('_', ' '))
    ax.grid(alpha=.18)
fig.suptitle('Matched-session release dispersion: common translation diagnostic')
fig.tight_layout()
fig.savefig(DIAG_FIG / '01_global_vs_pelvis_relative.png', dpi=170)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for axis, ax in zip(('y_sd', 'z_sd'), axes):
    d = primary_diag_correlations.query('axis == @axis').copy()
    d = d.iloc[:4].sort_values('rho')
    labels = d.comparison.str.replace(' / ', '\n/', regex=False)
    ix = np.arange(len(d))
    ax.errorbar(d.rho, ix, xerr=np.array([d.rho-d.ci_low, d.ci_high-d.rho]),
                fmt='o', capsize=3, color='#2b6b91')
    ax.set_yticks(ix, labels, fontsize=8)
    ax.axvline(0, color='gray', ls=':', lw=1)
    ax.set_title('Lateral Y' if axis == 'y_sd' else 'Vertical Z')
    ax.set_xlabel('Spearman rho; 95% bootstrap interval')
    ax.grid(axis='x', alpha=.2)
fig.suptitle('Axis-by-axis matched correlation sensitivity')
fig.tight_layout()
fig.savefig(DIAG_FIG / '02_lateral_vertical_associations.png', dpi=170)
plt.show()

chart = focused_cv.pivot(index='model', columns='target', values='mae_cm').loc[list(DIAG_FEATURES)]
fig, ax = plt.subplots(figsize=(10.5, 5.7))
ix = np.arange(len(chart))
w = .37
ax.barh(ix - w / 2, chart.global_hand, height=w, label='Global hand spread')
ax.barh(ix + w / 2, chart.pelvis_relative_hand, height=w, label='Pelvis-relative hand spread')
ax.set_yticks(ix, chart.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Repeated nested CV MAE (cm) | lower is better')
ax.set_title('Same pitchers, two distinct outcome definitions')
ax.legend()
ax.grid(axis='x', alpha=.2)
fig.tight_layout()
fig.savefig(DIAG_FIG / '03_matched_cv_models.png', dpi=170)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (metric, color) in zip(axes, (('rho_abs_to_global', '#2b6b91'),
                                      ('rho_rel_to_global', '#bc7037'))):
    ax.hist(lopo_associations[metric], bins=12, color=color, edgecolor='white', alpha=.85)
    ax.axvline(full[main_labels.index(metric)], ls=':', c='black', lw=1.5, label='Complete sample')
    ax.set_title(metric.replace('_', ' '))
    ax.set_xlabel('Leave-one-pitcher-out Spearman rho')
    ax.set_ylabel('Number of omitted-pitcher runs')
    ax.legend(fontsize=8)
fig.suptitle('Are the apparent relationships driven by one pitcher?')
fig.tight_layout()
fig.savefig(DIAG_FIG / '04_leave_one_pitcher_out_rho.png', dpi=170)
plt.show()

In [ ]:
manifest = dict(study='15_head_trunk_release_consistency', dataset='OpenBiomechanics Dataset v1',
                date_created='2026-10-17', primary_min_pitches=PRIMARY_MIN_PITCHES,
                min_pitch_sensitivities=SENSITIVITY_MIN_PITCHES,
                primary_head_lowpass_hz=PRIMARY_HEAD_CUTOFF_HZ,
                head_filter_sensitivities=FILTER_SENSITIVITY_HZ,
                pre_release_window_ms=PRE_RELEASE_MS, pre_release_grid_samples=len(TIME_GRID_MS),
                max_marker_gap_frames=MAX_MARKER_GAP_FRAMES,
                max_head_pair_geometry_change_mm=1000 * MAX_HEAD_PAIR_DISTANCE_CHANGE_M,
                outcome='sample-SD-based RMS of hand-joint-center Y and Z release positions (cm)',
                predictor_definition='pitcher-session sample-SD-based variability of release-time torso angles and head centroid positions',
                raw_trial_join='exact metadata.filename_new, verified against processed time grid',
                n_pitch_trials_requested=int(len(pitches)),
                n_complete_trials=int(len(pitch_metrics)),
                n_primary_sessions=int(len(sessions)),
                n_primary_pitchers=int(sessions.user.nunique()),
                bootstrap_reps=BOOTSTRAPS, group_cv_seeds=CV_SEEDS,
                inference_limits=[
                    'No observed pitch target or location; release consistency does not establish command.',
                    'Hand joint center is not the actual baseball location.',
                    'Small numbers of trials per session make session dispersion estimates noisy.',
                    'Only fastballs and one assessment session for most pitchers.',
                    'Associations are observational; model comparisons are exploratory, not causal.',
                    'Head centroid tracks translation, not gaze or independently validated head orientation.',
                    'Trajectory descriptors include release position and can be correlated with release-time metrics.',
                ])
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved derived CSVs and audit to:', OUT)
print('Saved figures to:', FIG)
print('Notebook execution does not modify GitHub.')

In [ ]:
diagnostic_manifest = dict(
    study='15_head_trunk_release_consistency_v2',
    source='User-run notebook 15 plus focused pelvis-relative diagnostic',
    dataset='OpenBiomechanics Dataset v1',
    source_release_position='processed hand joint center, not tracked baseball',
    pelvis_reference='arithmetic centroid of C3D LASI/RASI/LPSI/RPSI, 12 Hz filtered',
    head_reference='arithmetic centroid of C3D LFHD/RFHD/LBHD/RBHD, 12 Hz filtered',
    pelvis_position='head_absolute - (head-pelvis) derived from the preexisting extraction',
    note='All global C3D and processed landmark positions assumed in shared OBP laboratory frame',
    primary_min_pitch_count=PRIMARY_MIN_PITCHES,
    min_pitch_sensitivities=DIAG_SENSITIVITY_MIN,
    pitch_extractions=int(len(pitch_metrics)),
    pelvis_relative_pitch_extractions=int(len(focused_pitches)),
    primary_matched_sessions=int(len(focused)),
    primary_cv_sessions=int(len(complete_diag)),
    correlation_bootstraps=DIAG_BOOTSTRAPS,
    multiple_comparison_note='FDR for descriptive screen only; comparisons exploratory and correlated',
    results_directory=str(DIAG),
    figures_directory=str(DIAG_FIG),
    limitations=[
        'Within-session associations do not establish causal command or pitch location accuracy.',
        'Actual baseball location and target are unavailable; release proxy is hand joint center.',
        'Absolute head and hand positions can share global body movement.',
        'Head and hand both relative to pelvis share the same subtraction, inducing mathematical coupling.',
        'Pelvis marker centroid is not anatomical pelvis center or a directly measured hip joint center.',
        'C3D and processed hand-joint-center coordinate frames assumed common; examine separation audit.',
        'Only 2 to 5 pitches per session and one assessment per pitcher in Dataset v1.',
        'Axis-specific and model contrasts are exploratory and can vary with population or session.',
        'Leave-one-pitcher-out adjusted regression is re-fit; CV omission assesses held-out error influence, not CV re-fitting.'
    ])
(DIAG / '15_focused_diagnostic_manifest.json').write_text(json.dumps(diagnostic_manifest, indent=2), encoding='utf-8')
print('Focused diagnostic output:', DIAG)
print('Focused diagnostic visualizations:', DIAG_FIG)
print('Notebook execution does not modify GitHub.')